# 05. embedding: from 104 lipids to a handful of programs

You arrive with two coronal sections of mouse brain at the same anteroposterior plane, one from a
control (naive) female and one from a pregnant female, each pixel carrying intensities for the same
panel of lipids. The previous notebook handed you the normalized version of those intensities. The
table is wide: every pixel is a point in a 104-dimensional space, and you cannot look at 104
dimensions and see structure.

This notebook turns those 104 numbers per pixel into a handful of interpretable **lipid programs**, and
in doing so walks the backbone every spatial atlas rests on:

> feature selection by **Moran's I** -> **NMF** (the embedding) -> **t-SNE** (just to look) -> **Harmony** (only for clustering)

We do each step transparently: a few lines of plain code so you see the gears turn, then the matching
one-liner from the course helper `cajal_lipidomics`. The helper is the same recipe, made robust. The
plots are the teaching. We look at the data before, during, and after every transformation, because a
number you cannot see is a number you cannot trust.

Where this notebook sits in the chain: it loads `data/derived/03_normalized.h5ad` (the normalized
output of notebook 03) and saves `data/derived/05_embedded.h5ad`, which carries the NMF, t-SNE, and
Harmony embeddings into notebook 06, where you cluster them into your own lipizones.

Four markers run through the notebook:

- 🔬 **TASK**: something you do (write or run code).
- 💡 **HINT**: a nudge when you are stuck.
- ❓ **QUESTION**: pause and think, no code required.
- **check:** what you should see if it worked. If your screen disagrees, stop and reconcile it.

## the tools

We pull in the scientific-Python stack, the course helper package `cajal_lipidomics` (imported as
`cl`), and one global seed so every number and figure below is reproducible. `set_style()` applies the
course figure defaults.

In [ ]:
# 🔬 your code here


check: a single `ready: numpy ... | pandas ... | anndata ...` line and no red error. A
`ModuleNotFoundError` means the wrong kernel: pick `cajal-lipidomics` in the top right and rerun.

## load the normalized sections

We load the normalized stage into an **AnnData** object, the standard container for this kind of data.
Three parts always travel together:

- `adata.X` holds the **raw ion intensities** straight from METASPACE, one row per pixel, one column
  per detected ion. We keep them around for reference, but we do not embed on them.
- `adata.layers["umaia"]` holds the **uMAIA-normalized** intensities, the output of notebook 03. These
  are the values where the same lipid is finally comparable across the two sections, so this is what we
  build the embedding from.
- `adata.obs` is the **per-pixel metadata**: which section a pixel came from (`SectionID`), its
  condition (`Condition`), its position in the Allen common coordinate framework (`zccf`, `yccf`), and
  the Allen brain region it falls in (`acronym`) with that region's colour (`allencolor`).

`adata.var` carries the chemistry of each column: the molecular formula (the `var_names`), the m/z, and
a human-readable `lipid` name where our reference table could assign one. One pixel is one MALDI laser
spot, not one cell: it averages membranes across a small patch of tissue. Keep that in mind throughout.

In [ ]:
adata = ad.read_h5ad("../../data/derived/03_normalized.h5ad")

# the embedding features: uMAIA-normalized intensities, each lipid rescaled to its own 0-1 range so
# different lipids become comparable. This is exactly analysis.min01_per_lipid, the recipe from NB03.
umaia = adata.layers["umaia"]
Xn = analysis.min01_per_lipid(umaia)

# human-readable label per column; unmatched ions keep an "ion_<mz>" placeholder
names = adata.var["lipid"].astype(str).to_numpy()
n_annot = int((~pd.Series(names).str.startswith("ion_")).sum())

print("pixels x lipids:", adata.shape)
print("conditions:", dict(adata.obs["Condition"].value_counts()))
print("sections:", dict(adata.obs["SectionID"].value_counts()))
print(f"annotated columns: {n_annot} / {adata.n_vars} (the rest are unnamed METASPACE ions)")
print("a few names:", list(names[:6]))

check: `pixels x lipids: (189011, 104)`, two conditions (`naive` 88,753 pixels, `pregnant`
100,258), two sections (75 is the control 217D, 110 is the pregnant Brain1_C2), and 63 of the 104
columns carry a real lipid name. The other 41 are genuine ions METASPACE detected confidently but our
reference table could not name. They are not junk, they simply lack a label, and several of them turn
out to carry strong spatial signal.

Before any analysis, look at the raw data. We map one lipid across both sections so you see what a
single column looks like spread over the tissue. `HexCer 42:2` is a hexosylceramide, a sphingolipid
that coats **myelin**, the fatty insulation around axons that fills white matter. If everything is
wired correctly it should paint the fibre tracts.

We will scatter pixel values on the CCF coordinates several times, so we write one small helper
once. It plots `zccf` against `-yccf` (the brain upright), clips the colour scale to the 2nd and 98th
percentiles so a few hot pixels do not wash everything out, and strips the axes.

In [ ]:
def spatial_map(ax, obs, values, vmin=None, vmax=None, cmap="plasma", s=2):
    values = np.asarray(values)
    if vmin is None: vmin = np.quantile(values, 0.02)
    if vmax is None: vmax = np.quantile(values, 0.98)
    sc = ax.scatter(obs["zccf"], -obs["yccf"], c=values, cmap=cmap, s=s,
                    vmin=vmin, vmax=vmax, rasterized=True)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    return sc

lip = "HexCer 42:2"
j = int(np.where(names == lip)[0][0])           # column of the myelin lipid
v = Xn[:, j]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (cond, m) in zip(axes, [("naive", adata.obs["Condition"] == "naive"),
                                ("pregnant", adata.obs["Condition"] == "pregnant")]):
    m = m.to_numpy()
    spatial_map(ax, adata.obs[m], v[m])
    ax.set_title(f"{cond}: {lip}", fontsize=FS["m"])
fig.suptitle("one myelin lipid, both sections", fontsize=FS["l"])
plt.tight_layout(); plt.show()

check: two brain-shaped scatters, control on the left, pregnant on the right. The bright pixels
trace the fibre tracts, the corpus callosum arching over the top, the internal capsule, the fimbria.
That bright skeleton is myelin. One lipid alone drawing recognisable anatomy is the whole promise of
spatial lipidomics, and it is exactly the property we now exploit to separate signal from noise.

## 1. feature selection: keep the lipids that paint anatomy

**the problem.** Of our 104 columns, some draw crisp anatomy like the myelin map you just saw, and some
are mostly measurement noise: speckle that flickers pixel to pixel with no spatial pattern. Feeding
noise into the embedding wastes factors on junk and blurs the real structure. So before we compress, we
**select features**: keep the columns that carry real spatial signal, drop the ones that do not.

**the idea.** A real biological signal is **spatially smooth**: if a pixel is high in some lipid, its
neighbours tend to be high too, because tissue is organised into regions, not random dots. Noise has no
such smoothness. So we need one number per lipid that answers "do nearby pixels have similar values?".
That number is **Moran's I**, the workhorse measure of spatial autocorrelation.

### what Moran's I actually computes

Let us build it from scratch so it is never a black box. For one lipid we have a value $x_i$ at every
pixel $i$, and the pixel coordinates $(z_i, y_i)$. Four moves.

**1. define who is a neighbour.** For each pixel, find its $k$ nearest pixels in space (we use
$k = 6$). That gives a **k-nearest-neighbour graph**: a network wiring each pixel to its 6 closest
neighbours. The weight $w_{ij}$ is 1 if $j$ is a neighbour of $i$, else 0.

**2. centre the values.** Subtract the mean, $\tilde{x}_i = x_i - \bar{x}$. Now positive means "above
average for this lipid", negative means "below".

**3. multiply each pixel by its neighbours.** For every edge in the graph, form the product
$\tilde{x}_i \, \tilde{x}_j$. If a pixel and its neighbour are both above average (or both below), the
product is **positive**; if one is above and the other below, it is **negative**. Sum over all edges.

**4. normalise.** Divide by the total variance $\sum_i \tilde{x}_i^2$ and scale by $N/W$ (pixels over
total edge weight):

$$ I \;=\; \frac{N}{W}\,\frac{\sum_{i}\sum_{j} w_{ij}\,(x_i-\bar{x})(x_j-\bar{x})}{\sum_i (x_i-\bar{x})^2}. $$

Read it as a correlation between a pixel and its neighbours. $I \approx 1$: neighbours look alike,
strong spatial pattern. $I \approx 0$: neighbours are unrelated, salt-and-pepper noise. The products in
the numerator are the dot products from linear algebra, here measuring agreement between a pixel and
the pixels around it.

🔬 **TASK.** Compute Moran's I for one lipid, by hand, in a few transparent lines. We do it on
the **control** section only, because we learn everything on the control brain and transfer later, just
as the atlas does. We use `scipy`'s `cKDTree` to find each pixel's 6 nearest spatial neighbours, then
assemble the formula term by term.

In [ ]:
# 🔬 your code here


💡 **HINT.** `cKDTree(coords).query(coords, k=7)` returns, for each pixel, the indices of its 7
nearest points sorted by distance. The first is the pixel itself (distance 0), so we slice off column 0
with `idx[:, 1:]` to keep the 6 genuine neighbours.

The helper `cl.analysis.morans_i(values, coords, k=6)` does exactly these four steps. Let us confirm it
matches, then run it over all 104 columns.

In [ ]:
# the helper reproduces our hand computation
moran_helper = analysis.morans_i(Xc[:, j], coords_ctrl, k=6)
print(f"hand-rolled: {moran_hex:.4f}   helper: {moran_helper:.4f}")

# score every column in the control section
moran_all = np.array([analysis.morans_i(Xc[:, c], coords_ctrl, k=6) for c in range(Xc.shape[1])])
moran_series = pd.Series(moran_all, index=[f"{names[i]} [{i}]" for i in range(len(names))]).sort_values()

print("\nlowest Moran's I (noisiest columns):")
print(moran_series.head(4).round(3).to_string())
print("\nhighest Moran's I (cleanest, most anatomical):")
print(moran_series.tail(4).round(3).to_string())

check: the helper matches the hand computation to four decimals. `HexCer 42:2` sits near the
very top (Moran around 0.74), while the lowest-Moran column sits around 0.135, weak but still
clearly above zero. Two things worth
noticing. First, several of the highest-Moran columns are **unnamed ions**: they carry crisp anatomy
even though our reference table never gave them a lipid name, a reminder that "unannotated" is not the
same as "noise". Second, the whole distribution skews high (nothing sits near zero), which tells us this is already a curated,
high-confidence panel rather than raw peaks.

Now the proof that Moran's I means what we claim. We put the **highest**-Moran column next to the
**lowest** as spatial maps. High Moran should look like anatomy, low Moran should look like static.

In [ ]:
best_i = int(np.argmax(moran_all))
worst_i = int(np.argmin(moran_all))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, c in zip(axes, [best_i, worst_i]):
    spatial_map(ax, adata.obs[ctrl_mask], Xc[:, c])
    ax.set_title(f"{names[c]}\nMoran's I = {moran_all[c]:.2f}", fontsize=FS["m"])
fig.suptitle("high Moran = real anatomy   vs   low Moran = noise", fontsize=FS["l"])
plt.tight_layout(); plt.show()

check: left, the high-Moran column draws clean, connected structure, you can read the anatomy;
right, the low-Moran column is a noisy haze with no coherent pattern. Same tissue, same pixels, wildly
different spatial coherence. That is what Moran's I quantifies, and why it is the right filter.

❓ **QUESTION.** The low-Moran column is not necessarily fake. It might be a real molecule that simply
is not regionally organised, or one measured near the detection limit so noise dominates. Either way it
carries no *spatial* structure for our regional analysis. Why is "no spatial structure" a good reason
to drop a feature before we look for spatial lipid programs?

### is 0.4 a principled line? a permutation null

We are about to throw away every column with Moran's I below 0.4, so we should know what Moran's I looks like when there is *genuinely no spatial structure*. Then 0.4 is not a number pulled from the air, it is a line drawn well above the noise floor. The clean way to find that floor is a **permutation null**, the single most useful idea for judging whether a statistic is real or luck. It comes back later in the course for class enrichment and for gene importances, so we build it once, here, by hand.

The logic has three moves. **First**, state the null hypothesis: "this lipid has no spatial organisation, its value at a pixel is unrelated to where the pixel sits." **Second**, *simulate* that null by **shuffling** the lipid's values across pixels. Shuffling keeps the exact same set of numbers, the same histogram, the same mean and variance, but it scrambles which value sits at which location, so by construction any spatial pattern is destroyed while everything non-spatial is kept. Recompute Moran's I on the shuffled values and you get one draw of "what Moran's I would be if the null were true". Do it a few hundred times and the spread of those numbers is the **null distribution**: the range of Moran's I that pure chance produces on this graph, with these values. **Third**, compare. The **empirical p-value** is simply the fraction of null draws that match or beat the value we actually observed: if almost no shuffle ever reaches our lipid's Moran's I, the pattern is not luck. We add one to numerator and denominator, $(1+\#\{I_{\text{null}}\ge I_{\text{obs}}\})/(1+B)$, so a p-value is never exactly zero (we only ran finitely many shuffles).

In [ ]:
# reuse the control kNN graph `idx` (k=6) built above; a Moran's I that takes the graph as given,
# so we can recompute it thousands of times cheaply during shuffling
def moran_from_graph(x, idx):
    xc = x - x.mean()
    denom = (xc ** 2).sum()
    if denom == 0:
        return 0.0
    num = (xc[:, None] * xc[idx]).sum()
    return (len(x) / idx.size) * (num / denom)

B = 300                                              # number of shuffles -> resolution of the null
hi_col = j                                           # HexCer 42:2, the anatomical column

# build a column that genuinely has NO spatial structure: take a real lipid's values and scramble
# them once across space. same numbers, same histogram, but any anatomy is destroyed by construction.
# this is our honest negative control, the thing a non-significant result should look like.
scrambled = rng.permutation(Xc[:, hi_col])

null = {}
obs = {}
pval = {}
for tag, x in [("high (HexCer 42:2)", Xc[:, hi_col]), ("scrambled control", scrambled)]:
    x = np.asarray(x, float).copy()
    obs[tag] = moran_from_graph(x, idx)
    draws = np.empty(B)
    for b in range(B):
        draws[b] = moran_from_graph(rng.permutation(x), idx)   # shuffle values, keep the graph
    null[tag] = draws
    pval[tag] = (1 + int((draws >= obs[tag]).sum())) / (1 + B)
    print(f"{tag:>20}: observed I = {obs[tag]:+.3f} | null mean {draws.mean():+.4f} "
          f"(sd {draws.std():.4f}) | empirical p = {pval[tag]:.4f}")

print(f"\nnull never climbs above {max(null['high (HexCer 42:2)'].max(), null['scrambled control'].max()):.3f}, "
      f"so the 0.4 cutoff sits far out in the tail of pure noise")

We plot the two null distributions with the observed Moran's I marked, and draw the 0.4 threshold as a vertical line, so you can read significance and the cutoff off one picture.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
for tag, col in [("high (HexCer 42:2)", "mediumpurple"), ("scrambled control", "gray")]:
    ax.hist(null[tag], bins=40, alpha=0.6, color=col, label=f"null: {tag}")
    ax.axvline(obs[tag], color=col, lw=2)
    ax.text(obs[tag], ax.get_ylim()[1] * 0.9, f" observed I={obs[tag]:.2f}",
            color=col, fontsize=FS["xs"], ha="left", va="top")
ax.axvline(0.4, color="black", ls="--", lw=1.5, label="0.4 threshold")
ax.set_xlabel("Moran's I"); ax.set_ylabel("count over shuffles")
ax.set_title("permutation null: what Moran's I does under no spatial structure", fontsize=FS["m"])
ax.legend(fontsize=FS["xs"])
plt.tight_layout(); plt.show()

check: both null distributions are narrow blobs hugging zero, exactly what "no spatial structure" should give, because a shuffled lipid has nothing left to make neighbours agree. The scrambled control's observed Moran's I sits *inside* its own null (we built it to have no anatomy), so its empirical p-value is large and we correctly fail to call it spatially structured. HexCer 42:2 lands far to the right of anything a shuffle ever produced, so its p-value is at the floor of $1/(B+1)$: its anatomy is not luck. And the 0.4 line falls in the empty gap between the null and the real signal, which is the honest justification for the cutoff. It keeps the lipids whose spatial coherence pure chance never reaches, and drops the ones indistinguishable from a shuffle. One caveat worth stating plainly: every real column in this curated panel, even the lowest-Moran one (around 0.135), still sits to the *right* of its own shuffled null, so it is weakly but genuinely structured. That is why 0.4 is a *strength* threshold, not a significance threshold: it is not separating real from fake, it is keeping only the columns whose spatial coherence is strong enough to anchor regional programs. That is the same null-distribution reasoning we will reuse when we test lipid-class enrichment and gene importances later in the course.

🔬 **TASK.** Apply the filter. The atlas used a permissive threshold of Moran's I > 0.4 to drop
the noisiest features before embedding. We do the same: keep every column above 0.4, remember which
ones survived. This is the feature-selected panel that feeds the embedding.

In [ ]:
# 🔬 your code here


In [ ]:
# eyeball the Moran's I distribution across all ions before trusting the threshold
moran_tbl = pd.Series(moran_all, index=names).sort_values(ascending=False)
print("Moran's I across the ions (summary):"); print(moran_tbl.describe().round(3).to_string())
print("\nmost spatially structured:\n", moran_tbl.head(8).round(3).to_string())
print("\nleast (salt-and-pepper):\n", moran_tbl.tail(5).round(3).to_string())

check: about 64 of the 104 columns survive. The dropped named lipids are the ones with weak
spatial structure (several `PC 34:1` copies, some phosphatidic acids and phosphatidylglycerols), plus
many of the unnamed ions. In the full atlas this same logic, combined with dropout and cross-section
variance criteria, trimmed thousands of raw peaks down to roughly a hundred clean lipids; with two
sections we lean on Moran's I alone, the single most important of those criteria. The threshold is a
choice you state and can defend, not a law.

## molecular modules: which lipids travel together

Before we compress, look at *why* compression is even possible. If every lipid moved on its own we could not summarise 64 columns with 12 programs. But lipids do not move independently: a lipid that coats myelin rises wherever myelin is, and so do the other myelin lipids, so they rise and fall **together** across pixels. Groups of co-varying lipids are exactly the **molecular modules** NMF will turn into programs.

We can see those modules directly, before any model, with a **lipid-lipid correlation heatmap**: correlate every selected lipid against every other across the control pixels, then reorder rows and columns so co-varying lipids sit side by side. Bright red blocks on the diagonal are modules, sets of lipids that rise together.

💡 **HINT.** Open `src/cajal_lipidomics/plotting.py` and read `lipid_lipid_corr` before you run it. It is a few honest lines: `np.corrcoef`, a `1 - |r|` distance, and an optimal-leaf-ordered linkage so the blocks line up. Reading it is faster than guessing what the picture means.

In [ ]:
# correlate the selected lipids across the CONTROL pixels, the same matrix we are about to factor.
# wrap them in a tiny AnnData so the helper can label rows/cols by lipid name.
# several ions share a lipid name, so suffix repeats to keep every row/col label unique
seen, uniq_names = {}, []
for nm in sel_names:
    seen[nm] = seen.get(nm, 0) + 1
    uniq_names.append(nm if seen[nm] == 1 else f"{nm} #{seen[nm]}")
ctrl_panel = ad.AnnData(Xc_sel.astype("float32"), var=pd.DataFrame(index=uniq_names))
ax, ordered = plotting.lipid_lipid_corr(ctrl_panel)
plt.show()

# put a number on the brightest red block: the most correlated pair of DISTINCT lipids
# (several ions share a lipid name, so we mask same-name pairs to avoid a trivial r = 1.0)
C = np.corrcoef(Xc_sel.T); np.fill_diagonal(C, 0)
C[sel_names[:, None] == sel_names[None, :]] = -9
i, k = np.unravel_index(np.argmax(C), C.shape)
print(f"most correlated lipid pair: {sel_names[i]} & {sel_names[k]}  (r = {C[i, k]:.2f})")

check: the heatmap is not a uniform wash, it has **blocks**, square red patches along the diagonal where a handful of lipids all correlate strongly with one another. Each block is a module of lipids that paint the same territory together (the tightest pair prints above at r around 0.95). Off the diagonal you also see blue patches, pairs that are anticorrelated because where one membrane type dominates another is excluded.

This is the structure NMF exploits, and it is literally how `seeded_nmf` starts: it clusters lipids by this same correlation, picks the most central lipid of each block as a seed program, and lets NMF refine from there. So the blocks you see now are, roughly, the programs you are about to get. We have looked at the answer before computing it, which is the right way to trust a method.

## 2. NMF: compress the panel into additive programs

We have a clean panel, but it is still dozens of columns wide. We want to compress it into a handful of
**lipid programs**, co-varying groups of lipids that move as a unit, so that downstream we cluster and
visualise a few interpretable numbers per pixel instead of dozens of raw intensities. The tool is
**non-negative matrix factorisation**, NMF.

### the linear algebra, from the ground up

Our data is a matrix $V$ with shape (pixels x lipids), every entry an intensity. NMF approximates it as
a product of two **smaller, non-negative** matrices:

$$ V \;\approx\; W \, H. $$

- $H$ has shape (programs x lipids). Each **row** is one program: a recipe of non-negative weights
  saying how much each lipid belongs to that program. One program might be "lots of HexCer and SM", a
  myelin recipe; another "lots of PC", a grey-matter membrane recipe.
- $W$ has shape (pixels x programs). Each **row** is one pixel, the entries saying **how much of each
  program that pixel expresses**, again all non-negative.

To rebuild one pixel's full profile you take its program activities (a row of $W$) and mix the recipes
(the rows of $H$) in those amounts: pixel $i$ is $\sum_k W_{ik}\,H_{k\cdot}$, a weighted sum of recipes.
Per lipid that is a **dot product**, a row of $W$ against a column of $H$. Linear algebra you already
know, doing biological work.

**why non-negative matters.** Because nothing is ever subtracted, every pixel is literally a *sum of
parts, each present in some non-negative amount*. You can never have "minus 3 units of the myelin
program". That is what makes the factors readable as biology. The classic demonstration is NMF on face
photographs: it discovers parts (a nose, an eyebrow, a mouth) because faces are built by adding parts,
not subtracting them.

### NMF versus PCA, the cousin you met before

Both compress a wide table into a few new coordinates, but they differ in one decisive way:

- **PCA** finds orthogonal directions of maximum variance. Its components freely mix lipids with **plus
  and minus signs**, so a component can say "more PC *minus* SM". Elegant, but biologically awkward: a
  negative amount of a lipid has no meaning.
- **NMF** forbids negatives everywhere. Its programs are **purely additive parts**. You lose PCA's
  orthogonality and its variance ranking, but you gain components you can almost name.

For *interpretation* ("which lipids define this territory?") NMF wins. For raw variance accounting PCA
wins. The atlas uses NMF precisely because the goal is interpretable lipid programs. We will not take
the sign difference on faith: in a moment we fit both on the same toy matrix and read the signs off the
printed arrays.

### a tiny worked example so the shapes are concrete

Before touching real data, factor a tiny 4-pixel x 3-lipid toy matrix into 2 programs, so you can see
$V$, $W$, and $H$ as small printed arrays and check by eye that $W H$ rebuilds $V$.

In [ ]:
from sklearn.decomposition import NMF

# 4 pixels, 3 lipids. Pixels 0-1 are "myelin-like" (high lipid C), pixels 2-3 "grey-like" (high A,B)
V = np.array([[0.1, 0.2, 9.0],
              [0.2, 0.1, 8.0],
              [7.0, 6.0, 0.3],
              [8.0, 7.0, 0.2]])

toy = NMF(n_components=2, init="nndsvda", random_state=SEED, max_iter=500)
W_toy = toy.fit_transform(V)
H_toy = toy.components_

print("V (pixels x lipids):\n", V)
print("\nW (pixels x programs), how much of each program per pixel:\n", W_toy.round(2))
print("\nH (programs x lipids), the two recipes:\n", H_toy.round(2))
print("\nW @ H reconstructs V:\n", (W_toy @ H_toy).round(2))

check: `W @ H` lands almost exactly on `V`. Read the rows of `W`: pixels 0 and 1 load on one
program, pixels 2 and 3 on the other, and the rows of `H` show one recipe dominated by lipid C and one
by lipids A and B. NMF rediscovered the two pixel types and the two recipes with no labels, just from
the requirement that everything be non-negative and add up.

Now the same toy `V` through PCA, so the sign difference is shown, not asserted. We print PCA's
`components_` next to NMF's recipes. PCA's loadings carry minus signs (it may say "high in lipid C
*minus* lipids A and B"); every weight in NMF's `H` is zero or positive (it can only *add* lipids). That
is the whole interpretability argument, in two printed arrays.

In [ ]:
from sklearn.decomposition import PCA

pca_toy = PCA(n_components=2, random_state=SEED).fit(V)

print("NMF recipes H (programs x lipids), the additive parts:\n", H_toy.round(2))
print("every NMF weight >= 0 ?", bool((H_toy >= 0).all()))
print("\nPCA components (components x lipids), free to be positive OR negative:\n",
      pca_toy.components_.round(2))
print("any PCA loading < 0 ?", bool((pca_toy.components_ < 0).any()))

### seeding: a smarter start than random

Plain NMF starts from a generic guess and can land in a poor local optimum, giving factors that drift
run to run. The atlas uses a **seeded** start: group the lipids by how they correlate, take the most
representative lipid of each group as the seed for one program, and initialise $W$ and $H$ from those
seeds. The intuition: lipids that rise and fall together belong to the same program, so we hand NMF
that structure up front instead of making it rediscover it from noise.

`cl.embedding.seeded_nmf` implements exactly this:

1. compute the lipid-lipid correlation matrix and turn it into a distance ($1 - |\text{corr}|$, the
   absolute value because anticorrelated lipids still share structure),
2. cluster the lipids into `n_factors` groups by that distance,
3. pick the most central lipid of each group as a seed program,
4. run scikit-learn's `NMF` with `init="custom"` from those seeds.

💡 **HINT.** Open `src/cajal_lipidomics/embedding.py` and read `seeded_nmf`, `apply_nmf`, and `harmonize` before you call them in this section. They are short: `seeded_nmf` is the four steps above, `apply_nmf` is a single `model.transform`, and `harmonize` is a thin wrapper over `harmonypy`. Reading them is how you know exactly what each one-liner below actually does.

🔬 **TASK.** Learn the embedding on the **control** section's selected columns. We ask for 12 programs.
The atlas chose its factor count data-driven from the number of lipid correlation communities; 12 is a
sensible fixed choice for two sections that keeps every factor easy to inspect.

In [ ]:
# 🔬 your code here


check: `W` is (88753 x 12), one row per control pixel with its 12 program activities; `H` is
(12 x 64), one row per program with its lipid recipe; every entry is non-negative. Dozens of lipid
columns are now 12 program columns per pixel.

You may also see a red `ConvergenceWarning: Maximum number of iterations 400 reached`. That is expected
and harmless. NMF keeps polishing $W$ and $H$ by tiny amounts long after the factorisation has settled
into its shape, so scikit-learn warns that it stopped at the iteration cap rather than at a perfectly
flat optimum. The programs are already stable; pushing `max_iter` higher shifts the numbers by less
than a percent and changes no conclusion.

Now interpret programs in both directions: the **recipe** (a row of $H$, the top lipids) and the
**spatial map** (a column of $W$, painted over the tissue). A program is only useful if the two views
agree, a coherent set of lipids lighting up a coherent piece of anatomy.

In [ ]:
# rank programs by how much spatial signal they carry (variance of their per-pixel activity)
prog_var = W_ctrl.var(axis=0)
prog_order = np.argsort(prog_var)[::-1]

fig, axes = plt.subplots(3, 2, figsize=(11, 11),
                         gridspec_kw={"width_ratios": [1.1, 1.0]})
for r, p in enumerate(prog_order[:3]):
    top = np.argsort(H[p])[::-1][:8]                 # 8 highest-weighted lipids of this program
    axes[r, 0].barh(range(8), H[p][top][::-1], color="mediumpurple")
    axes[r, 0].set_yticks(range(8))
    axes[r, 0].set_yticklabels(sel_names[top][::-1], fontsize=FS["xs"])
    axes[r, 0].set_title(f"program {p}: lipid recipe (H)", fontsize=FS["m"])
    axes[r, 0].set_xlabel("weight")

    sc = spatial_map(axes[r, 1], adata.obs[ctrl_mask], W_ctrl[:, p], vmin=0)
    axes[r, 1].set_title(f"program {p}: activity per pixel (W)", fontsize=FS["m"])
plt.tight_layout(); plt.show()

check: for each of the three strongest programs the recipe and the map line up. A program built
from particular phosphatidylcholines paints one set of territories; a program built from other lipids
paints a different set. The activity maps are spatially coherent, not speckled, which is the payoff of
having filtered on Moran's I first. Each program is a molecular signature you could hand a biologist:
these lipids, here.

💡 **HINT.** We sorted programs by `W.var(axis=0)`, the spread of a program's activity across pixels. A
program that is roughly constant everywhere carries little spatial information and ranks low; one that
is high in some regions and low in others ranks high. Same logic as Moran's I and PCA: structure lives
in variance.

### apply the learned embedding to both sections

The atlas learns NMF on one brain and then **applies** it everywhere, so every pixel in every brain is
described in the *same* 12-program vocabulary. That shared vocabulary is what later lets us compare
control and pregnant pixels and transfer labels between them. Applying NMF means: keep $H$ (the recipes)
fixed and solve for each new pixel's program activities $W$. Scikit-learn does this with
`model.transform`, wrapped by `cl.embedding.apply_nmf`.

🔬 **TASK.** Project all 189,011 pixels (both sections) into the 12-program space the control defined,
and store the result in `adata.obsm["X_nmf"]`.

In [ ]:
# 🔬 your code here


check: the embedding is (189011 x 12), and on the control rows every factor correlates at
essentially 1.0 with the $W$ we learned, as it must, since applying the fixed recipes to the same
pixels reproduces their activities. Every pixel in both sections now lives in the same 12-dimensional
program space. From here on we work in that space, not in raw lipids.

## 3. t-SNE: a flat picture, for looking only

We have 12 program dimensions per pixel. Screens are flat. To *see* the structure we squash those 12
dimensions into 2 with a **neighbourhood-preserving** map: **t-SNE**. It asks "who are each pixel's near
neighbours in the 12-D program space?" and arranges the 2-D points so those same pixels stay close,
letting faraway pixels fall where they may. The result is a scatter where tight blobs are genuinely
similar pixels.

**We always run t-SNE on the embedding, never on raw lipids.** Three reasons: it is far faster on 12
columns than on dozens, the embedding has already denoised the data, and raw high-dimensional distances
are unreliable (in many dimensions everything drifts roughly equidistant, the "curse of
dimensionality"). Compress first with NMF, *then* look. That ordering is the spine of the pipeline.

**a warning to internalise.** A t-SNE plot is **for looking, not for measuring.** It preserves
who-is-near-whom locally, not global geometry. So:

- the **distance** between two blobs is not a real distance, do not read "twice as far apart",
- the **size** of a blob is not a real density, t-SNE inflates sparse groups and shrinks dense ones,
- the **axes** have no units and no meaning.

Use it to spot groups and check that biology separates. For any quantitative claim, go back to the
numbers: the program activities, the differential test, the cluster compositions.

🔬 **TASK.** Run t-SNE on the NMF embedding with `openTSNE`, the exact tool the real pipeline uses. We
lay out all pixels (a couple of minutes), then store the coordinates in `adata.obsm["X_tsne"]`.

In [ ]:
# 🔬 your code here


For a readable scatter we draw a reproducible random subsample of 25,000 pixels (plotting all
189k overplots into a smear). We colour the *same* coordinates three ways: by **section** (is there a
batch split?), by **condition**, and by **Allen region** (does the layout respect anatomy?). The region
colours come straight from `allencolor`, the registration output, so they are an honest external label,
not something we fit.

In [ ]:
sub_idx = rng.choice(adata.n_obs, size=25_000, replace=False)
ts = X_tsne[sub_idx]
obs_sub = adata.obs.iloc[sub_idx]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))

# by section (the batch)
for s, c in zip(sorted(obs_sub["SectionID"].unique()), ["tab:orange", "tab:purple"]):
    m = (obs_sub["SectionID"] == s).to_numpy()
    axes[0].scatter(ts[m, 0], ts[m, 1], s=3, alpha=0.5, color=c, label=f"section {s:.0f}", rasterized=True)
axes[0].set_title("by section (the batch)"); axes[0].legend(markerscale=3)

# by condition
for cond, c in zip(["naive", "pregnant"], ["tab:blue", "tab:green"]):
    m = (obs_sub["Condition"] == cond).to_numpy()
    axes[1].scatter(ts[m, 0], ts[m, 1], s=3, alpha=0.5, color=c, label=cond, rasterized=True)
axes[1].set_title("by condition"); axes[1].legend(markerscale=3)

# by Allen region colour (the anatomy we hope the layout respects)
axes[2].scatter(ts[:, 0], ts[:, 1], s=3, c=obs_sub["allencolor"], alpha=0.6, rasterized=True)
axes[2].set_title("by Allen region (anatomy)")

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
plt.tight_layout(); plt.show()

check: the right panel breaks into coloured islands that follow Allen regions, so the embedding
genuinely separates anatomical territories. Look now at the left panel: the two sections largely
overlap, but inside several blobs you can see an orange part and a purple part sitting side by side
rather than fully interleaved. That residual split is the **batch effect**, the same biological
territory landing slightly apart only because it was measured on two different acquisitions. The middle
panel shows the same split by condition, because in a two-section design condition and section are one
and the same thing.

❓ **QUESTION.** If we clustered on this embedding right now, some clusters could form around "which
section a pixel came from" rather than "what kind of tissue it is". Why would that wreck a later
control-versus-pregnant comparison, and which panel above is the evidence we have something to fix?

## 4. harmony: make the sections overlap without erasing biology

**the problem.** Two sections, two acquisitions, two batches. Beyond real biology, each batch carries a
technical offset: a different day, a slightly different instrument state, tiny differences in matrix
deposition. In the embedding that offset is the residual split you just saw. We want pixels of the same
biological type to mix across sections, while genuinely different territories stay apart.

**the idea.** Harmony works directly on the **embedding coordinates** (`X_nmf`), never on raw lipids. It
iterates two moves: (1) softly cluster all pixels together, ignoring batch, then (2) within each soft
cluster, nudge each batch's pixels toward the cluster's shared centre, removing the part of the offset
that is purely "which section". Repeat, and biologically identical groups melt together while truly
different groups stay separate.

**course-critical distinction. read this twice.**

Harmony's batch-corrected embedding is used **only for clustering and label transfer**. It is the space
in which we will find lipizones and carry labels from control to pregnant, because there we *want* the
two sections to coembed.

It is **never** used for the differential test. When we ask "which lipids changed in pregnancy?" we go
back to the uMAIA-normalized, non-Harmonized values. The reason is sharp: with only two sections,
condition and batch are perfectly confounded, so Harmony cannot tell a real pregnancy-versus-control
difference from a technical section offset. Hand it the condition difference and it will happily erase
it as if it were noise. So Harmony cleans the space we *cluster* in, and the differential test runs on
the untouched data. The atlas states this explicitly: all differential testing was done on the uMAIA
output, with dimensionality reduction and harmonisation used solely for clustering.

🔬 **TASK.** Run Harmony on the NMF embedding, using **section** as the batch variable. The helper
`cl.embedding.harmonize` wraps `harmonypy` and returns the corrected embedding in the same
(pixels x programs) shape.

In [ ]:
# 🔬 your code here


To see the effect we lay out the **Harmonized** embedding with its own t-SNE on the same 25,000
pixels and colour by section, beside the pre-Harmony t-SNE from section 3. If Harmony did its job, the
two sections should interleave more thoroughly while the anatomical islands stay intact.

In [ ]:
tsne_h = TSNE(n_components=2, perplexity=30, n_jobs=8, random_state=SEED)
ts_h = np.asarray(tsne_h.fit(Z_harm[sub_idx]))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, (coords2d, ttl) in zip(axes, [(ts, "before Harmony: by section"),
                                       (ts_h, "after Harmony: by section")]):
    for s, c in zip(sorted(obs_sub["SectionID"].unique()), ["tab:orange", "tab:purple"]):
        m = (obs_sub["SectionID"] == s).to_numpy()
        ax.scatter(coords2d[m, 0], coords2d[m, 1], s=3, alpha=0.5, color=c,
                   label=f"section {s:.0f}", rasterized=True)
    ax.set_title(ttl, fontsize=FS["m"]); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(markerscale=3)
plt.tight_layout(); plt.show()

A picture can flatter or mislead, so we also measure the mixing in a number, and we measure it in
the **12-D embedding space** where clustering actually happens, not in the 2-D t-SNE. For each pixel we
look at its 15 nearest neighbours in the embedding and compute the fraction that come from the *other*
section. If batches are perfectly mixed, that fraction sits near 0.5 (a neighbour is a coin-flip between
sections); if batches are segregated, it sits near 0. Harmony should push the typical fraction up.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def cross_batch_fraction(emb, batch):
    nn = NearestNeighbors(n_neighbors=16).fit(emb)
    _, ind = nn.kneighbors(emb)
    ind = ind[:, 1:]                                  # drop self
    same = (batch[ind] == batch[:, None]).mean(axis=1)
    return 1.0 - same                                 # fraction of neighbours from the OTHER batch

batch = adata.obs["SectionID"].to_numpy()[sub_idx]
mix_before = cross_batch_fraction(W_all[sub_idx], batch)     # NMF embedding
mix_after = cross_batch_fraction(Z_harm[sub_idx], batch)     # Harmonized embedding

print("                                  BEFORE -> AFTER Harmony")
print(f"mean   cross-section neighbour fraction: {mix_before.mean():.3f} -> {mix_after.mean():.3f}")
print(f"median cross-section neighbour fraction: {np.median(mix_before):.3f} -> {np.median(mix_after):.3f}")
print("(0.5 = sections perfectly mixed locally; 0 = a pixel's neighbours are all its own section)")

check: the numbers move in the right direction and by a real amount. The median cross-section
neighbour fraction climbs from 0.267 to 0.333, a jump of about a quarter, and the mean rises from 0.327
to 0.340. So a typical pixel goes from having roughly one neighbour in four drawn from the other section
to one in three, distinctly closer to the 0.5 of perfect local mixing. That is the finding: the NMF
embedding already coembeds the two sections reasonably, but a real residual section offset remained, and
Harmony measurably reduced it. Crucially it did so without collapsing the anatomical islands, exactly the
behaviour we want: same-tissue pixels mix better across sections while genuinely different territories
stay apart. With a stronger batch effect (many sections, different acquisition days) the same call would
move these numbers further still. The principle holds regardless of the magnitude: we cluster on this
Harmonized space and test on the non-Harmonized data.

❓ **QUESTION.** Suppose we had instead handed Harmony `Condition` as the batch variable and it had
pushed this fraction all the way to 0.5. Why would that be a disaster for the pregnancy comparison, and
how does it connect to the rule that the differential test must run on the uMAIA values, not on anything
Harmony touched?

## 5. a first neural network: can the lipidome place a pixel in the brain?

Step away from the pipeline for one section and ask a question that tests how much information the lipidome really carries. In the foundations notebook you fit a **linear regression**, a straight-line map from inputs to a number. Here we ask a harder version: given a pixel's lipid intensities, can we predict **where in the brain it sits**, its CCF coordinates? Position is not a linear function of lipids (the same lipid can be high in two unrelated places), so a straight line will not do. We need a **nonlinear regressor**, and the simplest workhorse is a small **neural network** (a multilayer perceptron, MLP): stacked layers of weighted sums passed through a nonlinearity, trained to minimise prediction error. This is your first real machine-learning model, the same idea as the atlas's Lipid2Position network, shrunk to run on a CPU.

We predict the two **in-plane** axes, `yccf` (dorsoventral) and `zccf` (mediolateral), the same coordinates we have been plotting the brain on. We hold out 25% of pixels the model never sees while training, predict their position from lipids alone, and score with the **per-axis Pearson r** between predicted and true, the same r you met as the goodness-of-fit measure in foundations.

💡 **HINT.** Open `src/cajal_lipidomics/ml.py` and read `predict_position` before running it. It is short and honest: a `StandardScaler`, scikit-learn's `MLPRegressor` with a `(256, 128, 64)` hidden stack, a train/test split, and `pearsonr` per axis. Knowing those four moves is knowing exactly what the cell does.

In [ ]:
from cajal_lipidomics import ml

# a REAL generalization test, as in the atlas: TRAIN the MLP on the control section's LEFT hemisphere
# and DEPLOY it on the pregnant section's left hemisphere. one hemisphere only, so bilateral symmetry
# cannot let it cheat with a mirrored guess. target = in-plane CCF position (yccf, zccf).
mlp_adata = ad.AnnData(Xn.astype('float32'), obs=adata.obs.copy())
fit = ml.predict_position(mlp_adata, coord_keys=('yccf', 'zccf'),
                          train_cond='naive', test_cond='pregnant', hemisphere='left',
                          max_iter=300, random_state=SEED)
print(f"trained on {fit['n_train']} control-left pixels, deployed on {fit['n_test']} pregnant-left pixels")
for axis, r in fit['r'].items():
    print(f"  deployed Pearson r for {axis}: {r:.3f}")

true, pred = fit['true'], fit['pred']
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, i, axis in zip(axes, range(2), fit['coord_keys']):
    ax.scatter(true[:, i], pred[:, i], s=2, alpha=0.2, color='mediumpurple', rasterized=True)
    lim = [min(true[:, i].min(), pred[:, i].min()), max(true[:, i].max(), pred[:, i].max())]
    ax.plot(lim, lim, 'k--', lw=1)
    ax.set_xlabel(f'true {axis}'); ax.set_ylabel(f'predicted {axis}')
    ax.set_title(f'{axis}: r = {fit["r"][axis]:.2f}', fontsize=FS['m']); ax.set_aspect('equal')
fig.suptitle('train on control, deploy on pregnant: the lipidome predicts position', fontsize=FS['l'])
plt.tight_layout(); plt.show()

check: the `yccf` cloud hugs the diagonal tightly (r around 0.94), the network places a pixel along the dorsoventral axis almost exactly from its lipids alone. The `zccf` (mediolateral) axis is looser (r around 0.5), and the reason is biology, not a broken model: the brain is **bilaterally symmetric**, so a pixel in the left hemisphere and its mirror twin on the right carry nearly identical lipid profiles but opposite `zccf`. The network cannot separate them from lipids alone, so it splits the difference and the scatter widens around the midline. That a tiny CPU network recovers dorsoventral position this well is the headline: the lipidome is not just texture, it encodes location. This is the same regression-and-r logic from the foundations notebook, now nonlinear and doing real anatomical work, and it is the bridge to the machine-learning notebooks later in the course.

## save the embedded stage

Everything downstream (clustering into lipizones, transferring labels control to pregnant) consumes the
three embeddings we just built. We attach them to the AnnData, record which columns survived feature
selection and their Moran scores for transparency, and write `data/derived/05_embedded.h5ad`. The next
notebook loads exactly this file.

In [ ]:
adata.var["moran_i"] = moran_all.astype("float32")
adata.var["selected"] = keep_mask
adata.uns["embedding"] = {"n_factors": int(N_FACTORS), "moran_threshold": float(MORAN_THR),
                          "n_selected": int(keep_mask.sum())}

out = "../../data/derived/05_embedded.h5ad"
adata.write_h5ad(out)
print("wrote", out)
print("obsm now carries:", list(adata.obsm.keys()))

check: the file is written and `obsm` carries `X_nmf`, `X_tsne`, and `X_harmony`. That is the
handoff to notebook 06.

## what we built, and where it goes next

We took two coronal sections, 189,011 pixels by 104 columns, and walked the embedding pipeline end to
end, looking at the data at every step:

- **feature selection by Moran's I.** We built spatial autocorrelation from a kNN graph and four lines
  of arithmetic, proved with side-by-side maps that high Moran means real anatomy and low Moran means
  noise, and kept the columns above 0.4. Helper: `cl.analysis.morans_i`.
- **NMF.** We compressed the clean panel into 12 non-negative, additive lipid programs, showed on a toy
  matrix that NMF's recipes stay all-positive while PCA's components carry minus signs, read each
  program in two ways (its recipe $H$ and its spatial map $W$), and applied the control-learned factors
  to both sections so they share one vocabulary. Helpers: `cl.embedding.seeded_nmf`,
  `cl.embedding.apply_nmf`.
- **t-SNE.** We laid the embedding flat with `openTSNE`, saw the anatomy separate into islands, and kept
  the standing warning that distances, blob sizes, and axes carry no quantitative meaning. The layout
  also exposed a residual section split.
- **Harmony.** We reduced that section offset in the embedding space, kept the territories distinct, and
  measured the mixing in a number. We stated, twice, that Harmony is for clustering and label transfer
  only, never for the differential test, which always runs on the untouched uMAIA values. Helper:
  `cl.embedding.harmonize`.

The Harmonized embedding `adata.obsm["X_harmony"]` is the input to the next notebook, where you cluster
it into **your own lipizones** and transfer those labels from control to pregnant. Your lipizones will
not match the paper's exactly, and that is the honest, realistic outcome of learning the structure from
two sections rather than a whole atlas. The broad biology still emerges. The differential test that
follows will reach back past Harmony to the uMAIA values. Two paths, kept clean, all the way through.